## Experiment 7 — Bounds-Check Failure Demo



**Objective:**
Demonstrate why CUDA kernels require bounds checks when the total number of launched threads exceeds the number of valid data elements.

- Run a kernel with and without `if (idx < N)`.
- Observe the effect of out-of-bounds memory access.
- Understand why ceiling-divided grid sizes create extra threads in the final block.
- Reinforce safe CUDA indexing practices.

## Implementation

In [1]:
!nvidia-smi

Wed Sep  9 06:34:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             31W /   70W |     447MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install ninja

In [3]:
# import libraries
import torch # for pytorch
from torch.utils.cpp_extension import load_inline # for loading cuda/c++ code

# versions
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")


PyTorch version: 2.11.0+cu128
CUDA available: True
cuDNN version: 91900


In [6]:
cpp_code = """
#include <torch/extension.h>

torch::Tensor vector_add_launcher(torch::Tensor input1, torch::Tensor input2, std::string mode);
"""

cuda_code = """
#include <torch/extension.h>

// This kernel will fail to execute if the total number of threads exceeds the number of valid data elements.

__global__ void vector_add_kernel(const float* input1, const float* input2, float* output, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    output[idx] = input1[idx] + input2[idx];
}

// This kernel will execute correctly even if the total number of threads exceeds the number of valid data elements.

__global__ void vector_add_kernel_safe(const float* input1, const float* input2, float* output, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        output[idx] = input1[idx] + input2[idx];
    }
}

torch::Tensor vector_add_launcher(torch::Tensor input1, torch::Tensor input2, std::string mode) {
    int N = input1.numel();
    torch::Tensor output = torch::zeros_like(input1, input1.options());
    if (mode == "unsafe") {
        vector_add_kernel<<<1, N>>>(input1.data_ptr<float>(), input2.data_ptr<float>(), output.data_ptr<float>(), N);
    } else {
        vector_add_kernel_safe<<<1, N>>>(input1.data_ptr<float>(), input2.data_ptr<float>(), output.data_ptr<float>(), N);
    }
    return output;
}

"""


In [7]:
module = load_inline(
    name="vector_add",
    cpp_sources=cpp_code,
    cuda_sources=cuda_code,
    functions=["vector_add_launcher"],
    extra_cuda_cflags=["-O3"],
    verbose=True
)




In [8]:
input1 = torch.randn(256, device="cuda")
input2 = torch.randn(256, device="cuda")



In [9]:
output_safe = module.vector_add_launcher(input1, input2, "safe")
print(output_safe)




tensor([-8.2345e-01,  2.1015e+00,  1.8775e+00, -1.7586e-01,  7.8368e-01,
         3.2970e-01,  8.9123e-01,  5.8962e-01, -7.3553e-01,  1.2856e+00,
        -2.1950e+00, -1.1306e+00, -1.6083e+00, -1.8920e+00, -7.9348e-01,
        -2.9276e-01,  3.6177e-01,  2.9504e-01,  1.8601e-01,  1.2010e+00,
        -9.0122e-01, -1.9720e+00, -4.6934e+00, -8.4252e-01,  1.6336e+00,
        -1.2017e+00,  1.1052e+00,  7.5088e-01,  4.5439e-01, -2.3601e+00,
         1.2173e+00,  1.0457e-01,  3.0058e-01, -1.8858e+00, -1.9861e-01,
         8.9744e-01, -3.5592e-01, -2.1370e+00,  1.1311e+00,  4.0568e+00,
        -1.1326e+00, -1.0305e+00,  9.4505e-01, -7.7852e-01,  2.0628e+00,
         1.2106e+00, -1.5741e-01,  2.6155e+00,  1.1599e+00, -1.9918e+00,
         6.3934e-01, -3.5593e+00,  3.4675e-01, -3.3390e-01, -1.3136e-01,
        -1.9387e+00, -1.8801e+00, -1.2319e+00,  4.2607e-01, -4.5726e-01,
         5.9380e-01,  9.6994e-01, -6.4151e-01,  5.4334e-01,  2.1133e-01,
        -1.5928e+00, -7.4173e-02, -3.3132e-02,  5.8

In [10]:
output_unsafe = module.vector_add_launcher(input1, input2, "unsafe")
print(output_unsafe)

tensor([-8.2345e-01,  2.1015e+00,  1.8775e+00, -1.7586e-01,  7.8368e-01,
         3.2970e-01,  8.9123e-01,  5.8962e-01, -7.3553e-01,  1.2856e+00,
        -2.1950e+00, -1.1306e+00, -1.6083e+00, -1.8920e+00, -7.9348e-01,
        -2.9276e-01,  3.6177e-01,  2.9504e-01,  1.8601e-01,  1.2010e+00,
        -9.0122e-01, -1.9720e+00, -4.6934e+00, -8.4252e-01,  1.6336e+00,
        -1.2017e+00,  1.1052e+00,  7.5088e-01,  4.5439e-01, -2.3601e+00,
         1.2173e+00,  1.0457e-01,  3.0058e-01, -1.8858e+00, -1.9861e-01,
         8.9744e-01, -3.5592e-01, -2.1370e+00,  1.1311e+00,  4.0568e+00,
        -1.1326e+00, -1.0305e+00,  9.4505e-01, -7.7852e-01,  2.0628e+00,
         1.2106e+00, -1.5741e-01,  2.6155e+00,  1.1599e+00, -1.9918e+00,
         6.3934e-01, -3.5593e+00,  3.4675e-01, -3.3390e-01, -1.3136e-01,
        -1.9387e+00, -1.8801e+00, -1.2319e+00,  4.2607e-01, -4.5726e-01,
         5.9380e-01,  9.6994e-01, -6.4151e-01,  5.4334e-01,  2.1133e-01,
        -1.5928e+00, -7.4173e-02, -3.3132e-02,  5.8

## Observation



- The unsafe kernel still produced an output even though extra threads accessed memory beyond the valid tensor bounds.
- Out-of-bounds CUDA accesses do not always cause an immediate crash or visible error.
- This makes unsafe kernels particularly dangerous because memory corruption can occur silently.
- Bounds checks such as `if (idx < N)` are therefore necessary whenever the grid size may launch more threads than there are valid elements.